# Flight Ticket Price Prediction — MLP Kaggle Assignment


This notebook is organised into the following sections:

1. Setup & Data Loading
2. Dataset Understanding
3. Descriptive Statistics
4. Missing Value Analysis & Handling
5. Duplicate Row Analysis
6. Outlier Analysis
7. Exploratory Data Analysis (EDA)
8. Feature Engineering
9. Preprocessing (Pipelines: Scaling + Encoding)
10. Model Building (7+ models, 5-fold CV)
11. Hyperparameter Tuning (3 models)
12. Model Comparison
13. Train Final Model on Full Data
14. Kaggle Submission
15. Feature Importance
16. Conclusion


## 1. Setup & Data Loading

We first import all the libraries we will need throughout the notebook, then load the three
provided files: `train.csv`, `test.csv`, and `sample_submission.csv`.



In [ ]:
# Core libraries
import os
import glob
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')

# Plot styling
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (8, 5)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print('Libraries imported successfully.')


In [ ]:
import pandas as pd

DATA_PATH = "/kaggle/input/competitions/mlp-may-2026-kaggle-assignment-1"

train_df = pd.read_csv(f"{DATA_PATH}/train.csv")
test_df = pd.read_csv(f"{DATA_PATH}/test.csv")
sample_submission = pd.read_csv(f"{DATA_PATH}/sample_submission.csv")

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("Sample submission shape:", sample_submission.shape)

train_df.head()


## 2. Dataset Understanding

Before doing anything else, we need to understand what the data actually looks like: how many
rows/columns there are, what each column represents, and what data type each column has.


In [ ]:
# First few rows of the training data
train_df.head()


In [ ]:
# Basic shape information
print(f'Training set : {train_df.shape[0]} rows, {train_df.shape[1]} columns')
print(f'Test set     : {test_df.shape[0]} rows, {test_df.shape[1]} columns')


In [ ]:
# Full structural summary: column names, non-null counts, and dtypes
train_df.info()


In [ ]:
# Explicit data types of every column (rubric requirement)
dtype_table = pd.DataFrame({
    'column': train_df.columns,
    'dtype': train_df.dtypes.astype(str).values
})
dtype_table


**Column meanings (based on the values observed):**

| Column | Type | Description |
|---|---|---|
| `id` | int | Unique row identifier |
| `airline` | categorical | Airline operating the flight |
| `flight` | categorical (high-cardinality) | Flight number/code |
| `source` | categorical | Departure city |
| `departure` | categorical | Time-of-day bucket for departure (Morning, Evening, ...) |
| `stops` | categorical (ordinal-ish) | Number of stops (zero / one / two_or_more) |
| `arrival` | categorical | Time-of-day bucket for arrival |
| `destination` | categorical | Arrival city |
| `class` | categorical | Economy or Business |
| `duration` | numerical (float) | Flight duration in hours |
| `days_left` | numerical (float) | Days left until departure at the time of booking |
| `price` | numerical (int) — **target** | Ticket price (only in `train.csv`) |

Columns `airline`, `source`, `departure`, `stops`, `arrival`, `destination`, and `class` are
**categorical/text** columns (`str` dtype). `duration`, `days_left`, and `price` are
**numerical** columns (`float64`/`int64`). `id` and `flight` are identifier-like columns that
carry (almost) no predictive information on their own and will be handled during feature
engineering.


## 3. Descriptive Statistics

We now look at summary statistics (mean, median, standard deviation, min, max, quartiles) for
the numerical columns, and value counts for the categorical columns.


In [ ]:
# Descriptive statistics for numerical columns
numerical_cols = ['duration', 'days_left', 'price']

desc_stats = train_df[numerical_cols].describe().T
desc_stats['median'] = train_df[numerical_cols].median()
desc_stats = desc_stats[['count', 'mean', 'median', 'std', 'min', '25%', '50%', '75%', 'max']]
desc_stats


In [ ]:
# Descriptive statistics for categorical columns (frequency of most common category)
categorical_cols = ['airline', 'source', 'departure', 'stops', 'arrival', 'destination', 'class']

for col in categorical_cols:
    print(f'--- {col} ---')
    print(train_df[col].value_counts(dropna=False))
    print()


**Observations:**
- `price` is highly right-skewed: the mean (~20,800) is much higher than the median (~7,350),
  and the max (114,704) is far from the 75th percentile (~42,500). This is typical of ticket
  price data, where a smaller number of expensive Business-class tickets pull the mean up.
- `duration` ranges from under 1 hour to ~47 hours (likely including layovers for multi-stop
  flights), with a mean of ~12 hours.
- `days_left` ranges from 1 to 49 days before departure.
- `class` has only two categories (Economy/Business), which is very likely one of the strongest
  price predictors, since Business class tickets are usually priced far higher than Economy.


## 4. Missing Values

Next we identify missing values in both the training and test sets, and decide how to handle
them. Since the test set must be used for the final Kaggle submission, we **cannot drop rows**
from it — any missing values in `test.csv` must be imputed. For consistency (and because
dropping rows loses valuable training data), we will **impute** missing values in the training
set as well, rather than dropping them, using pipelines fitted only on the training data (to
avoid data leakage).


In [ ]:
# Missing values BEFORE handling
print('Missing values in TRAIN set:')
missing_train = train_df.isnull().sum()
missing_train_pct = (missing_train / len(train_df) * 100).round(2)
missing_train_table = pd.DataFrame({'missing_count': missing_train, 'missing_pct': missing_train_pct})
display(missing_train_table[missing_train_table['missing_count'] > 0].sort_values('missing_count', ascending=False))

print()
print('Missing values in TEST set:')
missing_test = test_df.isnull().sum()
missing_test_pct = (missing_test / len(test_df) * 100).round(2)
missing_test_table = pd.DataFrame({'missing_count': missing_test, 'missing_pct': missing_test_pct})
display(missing_test_table[missing_test_table['missing_count'] > 0].sort_values('missing_count', ascending=False))


**Findings:** Several columns have missing values in both the train and test sets:
`airline`, `departure`, `stops` (categorical) and `duration`, `days_left` (numerical). None of
the missing percentages are extreme enough (all are well under ~12%) to justify dropping the
columns entirely, so imputation is the right strategy:

- **Numerical columns** (`duration`, `days_left`): impute with the **median**, which is robust
  to the skewness/outliers we saw in the descriptive statistics.
- **Categorical columns** (`airline`, `departure`, `stops`): impute with the **most frequent
  category** (mode), which is the standard, safe default for categorical data.

To avoid data leakage, this imputation will be done **inside a scikit-learn `Pipeline`**
(Section 9) that is fit only on the training data and then applied to the test data — not by
manually filling both dataframes up front. We demonstrate the *before* state here, and show the
*after* state once the pipeline has been fit, later in the notebook.


In [ ]:
# Quick sanity check: confirm there are no missing values in the target column
print('Missing values in target (price):', train_df['price'].isnull().sum())


## 5. Duplicate Rows

Duplicate rows can bias a model by over-weighting certain samples. We check for, and remove,
exact duplicate rows in the training data. (We do not remove rows from the test set, since every
test row needs a prediction for the Kaggle submission.)


In [ ]:
# Check for duplicate rows in the training set
n_duplicates = train_df.duplicated().sum()
print(f'Number of duplicate rows found in train.csv: {n_duplicates}')

if n_duplicates > 0:
    before = train_df.shape[0]
    train_df = train_df.drop_duplicates().reset_index(drop=True)
    after = train_df.shape[0]
    print(f'Removed {before - after} duplicate rows. New shape: {train_df.shape}')
else:
    print('No duplicate rows found — nothing to remove.')


## 6. Outlier Analysis

We use boxplots together with the IQR (Interquartile Range) method to identify outliers in the
numerical columns.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, col in zip(axes, numerical_cols):
    sns.boxplot(y=train_df[col], ax=ax, color='steelblue')
    ax.set_title(f'Boxplot of {col}')

plt.tight_layout()
plt.show()


In [ ]:
def iqr_outlier_summary(df, col):
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    n_outliers = ((df[col] < lower) | (df[col] > upper)).sum()
    pct_outliers = n_outliers / df[col].notnull().sum() * 100
    return lower, upper, n_outliers, pct_outliers

print(f"{'Column':<12}{'Lower bound':>14}{'Upper bound':>14}{'#Outliers':>12}{'% Outliers':>12}")
for col in numerical_cols:
    lower, upper, n_out, pct_out = iqr_outlier_summary(train_df, col)
    print(f'{col:<12}{lower:>14.2f}{upper:>14.2f}{n_out:>12}{pct_out:>11.2f}%')


**Explanation — why we RETAIN the outliers:**

- **`price` outliers (high-priced tickets):** These are almost entirely genuine **Business
  class** tickets, not data errors. Business fares are legitimately several times more expensive
  than Economy fares, so these "outliers" represent a real, important sub-population that the
  model must learn to predict — not noise to be removed. We keep them, and instead rely on
  `class` as a strong feature to help the model distinguish these cases.
- **`duration` outliers (very long flights):** Long durations correspond to flights with
  **multiple stops/layovers**, which is realistic (not erroneous) behaviour in flight-search
  data. Removing them would remove legitimate multi-stop flight records.
- **`days_left` outliers:** Very few/no outliers here — bookings simply range from 1 to 49 days
  ahead, which is an expected, bounded quantity.

Since a Kaggle competition scores predictions on the **actual test distribution** (which will
also contain Business-class/multi-stop flights), removing these rows from training would only
hurt generalisation. Instead of deleting outliers, we use **tree-based models** (Decision Tree,
Random Forest, Extra Trees, HistGradientBoosting) which are naturally robust to outliers, and we
use **`StandardScaler`** for linear models, which reduces (but doesn't eliminate) the influence
of scale differences. No rows are dropped for outliers in this notebook.


## 7. Exploratory Data Analysis (EDA)

We now create several visualizations to better understand relationships between features and
the target variable `price`.


In [ ]:
# Visualization 1: Distribution of the target variable
plt.figure(figsize=(8, 5))
sns.histplot(train_df['price'], bins=50, kde=True, color='teal')
plt.title('Distribution of Flight Ticket Price')
plt.xlabel('Price')
plt.ylabel('Count')
plt.show()


**Insight:** The price distribution is strongly right-skewed with a large mass of
low-to-mid priced (Economy) tickets and a smaller, separated cluster of high-priced (Business)
tickets. This bimodal-looking shape strongly hints that `class` will be a key predictive
feature.


In [ ]:
# Visualization 2: Price by class and by airline
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.boxplot(data=train_df, x='class', y='price', ax=axes[0], palette='Set2')
axes[0].set_title('Price by Class')

order = train_df.groupby('airline')['price'].median().sort_values(ascending=False).index
sns.boxplot(data=train_df, x='airline', y='price', order=order, ax=axes[1], palette='Set3')
axes[1].set_title('Price by Airline')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()


**Insight:** Business class tickets are priced dramatically higher than Economy tickets,
confirming `class` as a major driver of price. Prices also vary noticeably by `airline` — some
airlines (e.g. Vistara, Air India, which offer Business class) show a much wider/higher price
range, largely because they are the airlines that sell Business class seats, while pure
budget carriers only sell Economy.


In [ ]:
# Visualization 3: Correlation heatmap of numerical features
plt.figure(figsize=(6, 5))
corr = train_df[numerical_cols].corr()
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f', vmin=-1, vmax=1)
plt.title('Correlation Heatmap (Numerical Features)')
plt.show()


**Insight:** `duration` has a weak positive correlation with `price`, and `days_left` has a
weak negative correlation with `price` (booking earlier tends to be slightly cheaper). Neither
numerical feature alone is strongly correlated with price — this tells us that the **categorical
features** (especially `class`) will contribute most of the predictive power, and that we may
need models capable of capturing non-linear/interaction effects (e.g. tree ensembles).


In [ ]:
# Visualization 4: Price vs. days_left, colored by class
plt.figure(figsize=(9, 6))
sample = train_df.sample(min(5000, len(train_df)), random_state=RANDOM_STATE)
sns.scatterplot(data=sample, x='days_left', y='price', hue='class', alpha=0.4, palette='Set1')
plt.title('Price vs. Days Left Before Departure (sampled)')
plt.show()


**Insight:** For both classes, prices tend to be highest when booking very close to the
departure date (low `days_left`) and gradually decrease as `days_left` increases, then flatten
out — a classic "last-minute booking premium" pattern seen in airline pricing.


In [ ]:
# Visualization 5: Number of flights by number of stops
plt.figure(figsize=(7, 5))
sns.countplot(data=train_df, x='stops', order=train_df['stops'].value_counts().index, palette='viridis')
plt.title('Flight Count by Number of Stops')
plt.xlabel('Stops')
plt.ylabel('Count')
plt.show()


**Insight:** The vast majority of flights in the dataset have **one stop**, followed by
**zero stops** (direct flights); flights with two-or-more stops are rare. This class imbalance
in `stops` is worth keeping in mind, though tree-based models generally handle it well.


## 8. Feature Engineering

Based on the EDA above, we perform the following feature engineering steps:

1. **Drop `id`** — a pure row identifier with no predictive value.
2. **Drop `flight`** — this column has 868 distinct values, and ~14% of the values are the
   placeholder string `"0.00E+00"` (a corrupted/blank flight code, likely from an Excel
   scientific-notation formatting bug on empty cells). Because it is extremely high-cardinality
   and partly corrupted, one-hot encoding it would create hundreds of extra sparse columns and
   add noise rather than signal (the same information — which airline operates the flight — is
   already captured cleanly by the `airline` column). We therefore drop it.
3. **Keep `stops` as a categorical feature** (zero / one / two_or_more) — it is already a clean,
   low-cardinality categorical column, so no further transformation is required beyond encoding.

No date columns are present in this dataset (only time-of-day buckets and `days_left`, which are
already numeric/categorical), so no date-parsing feature engineering is needed here.


In [ ]:
def engineer_features(df):
    """Apply the feature engineering steps decided above to a dataframe (train or test)."""
    df = df.copy()
    df = df.drop(columns=['id', 'flight'], errors='ignore')
    return df


train_fe = engineer_features(train_df)
test_fe = engineer_features(test_df)

print('Columns after feature engineering:')
print(list(train_fe.columns))


## 9. Preprocessing

We now build the preprocessing pipeline using scikit-learn's `Pipeline` and `ColumnTransformer`.

**Why scale numerical features?** Linear models (Linear Regression, Ridge, Lasso, ElasticNet)
and distance/gradient-based optimisation are sensitive to the *scale* of input features. Our
numerical features (`duration`, `days_left`) are on very different scales (hours vs. days), and
without scaling, features with larger raw magnitudes can dominate the model's coefficients and
slow down convergence. We use `StandardScaler` (zero mean, unit variance) inside the pipeline.

**Why encode categorical features?** Machine learning models in scikit-learn require numeric
input — they cannot directly interpret text categories like `"Vistara"` or `"Economy"`. We use
`OneHotEncoder` to convert each categorical column into a set of binary (0/1) indicator columns,
which lets models use categorical information without assuming any false ordinal relationship
between categories (e.g. there's no meaningful numeric order between "Delhi" and "Mumbai").

Both **imputation** and **transformation** are placed inside the same pipeline objects, and
fitted only on the training data, which prevents data leakage from the test set.


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PolynomialFeatures

numerical_features = ['duration', 'days_left']
categorical_features = ['airline', 'source', 'departure', 'stops', 'arrival', 'destination', 'class']

# --- Standard preprocessing pipeline (used by most models) ---
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numerical_features),
    ('cat', categorical_transformer, categorical_features)
])

print('Standard preprocessor created.')
preprocessor


In [ ]:
# --- Separate preprocessing pipeline for Polynomial Regression ---
# We expand ONLY the numerical features to degree=2 (interaction + squared terms),
# then scale them. Categorical features are still one-hot encoded as usual.
# (Applying PolynomialFeatures after one-hot encoding would explode the feature
#  space unnecessarily, since interactions between dummy variables are not meaningful.)
poly_numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('poly', PolynomialFeatures(degree=2, include_bias=False)),
    ('scaler', StandardScaler())
])

poly_preprocessor = ColumnTransformer(transformers=[
    ('num', poly_numeric_transformer, numerical_features),
    ('cat', categorical_transformer, categorical_features)
])

print('Polynomial preprocessor created.')
poly_preprocessor


In [ ]:
# Demonstrate missing-value handling: fit the standard preprocessor on the training
# data and confirm that the transformed output has NO missing values (i.e. imputation worked).
X_demo = preprocessor.fit_transform(train_fe[numerical_features + categorical_features])
print('Missing values in train set BEFORE pipeline:',
      train_fe[numerical_features + categorical_features].isnull().sum().sum())
print('Missing values in transformed output AFTER pipeline:', np.isnan(X_demo.toarray() if hasattr(X_demo, "toarray") else X_demo).sum())


## 10. Model Building

We train **nine** different scikit-learn regression models (well above the minimum of 7), each
wrapped in a `Pipeline` with the preprocessing step defined above. Every model is evaluated using
**5-fold cross-validation** on the training set, using two metrics:

- **R² (coefficient of determination)** — higher is better (1.0 = perfect).
- **RMSE (Root Mean Squared Error)** — lower is better, in the same units as `price`.


In [ ]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, HistGradientBoostingRegressor
from sklearn.model_selection import KFold, cross_validate

X = train_fe.drop(columns=['price'])
y = train_fe['price']

cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# Define all candidate models as (name, pipeline) pairs
models = {
    'Linear Regression': Pipeline([
        ('preprocessor', preprocessor),
        ('regressor', LinearRegression())
    ]),

    'Ridge Regression': Pipeline([
        ('preprocessor', preprocessor),
        ('regressor', Ridge(random_state=RANDOM_STATE))
    ]),

    'Lasso Regression': Pipeline([
        ('preprocessor', preprocessor),
        ('regressor', Lasso(random_state=RANDOM_STATE, max_iter=5000))
    ]),

    'ElasticNet': Pipeline([
        ('preprocessor', preprocessor),
        ('regressor', ElasticNet(random_state=RANDOM_STATE, max_iter=5000))
    ]),

    'Polynomial Regression (deg=2)': Pipeline([
        ('preprocessor', poly_preprocessor),
        ('regressor', LinearRegression())
    ]),

    'Decision Tree': Pipeline([
        ('preprocessor', preprocessor),
        ('regressor', DecisionTreeRegressor(
            random_state=RANDOM_STATE,
            max_depth=12
        ))
    ]),

    'Random Forest': Pipeline([
        ('preprocessor', preprocessor),
        ('regressor', RandomForestRegressor(
            n_estimators=50,
            max_depth=10,
            random_state=RANDOM_STATE,
            n_jobs=-1
        ))
    ]),

    'Extra Trees': Pipeline([
        ('preprocessor', preprocessor),
        ('regressor', ExtraTreesRegressor(
            n_estimators=100,
            random_state=RANDOM_STATE,
            n_jobs=-1
        ))
    ]),
}

print(f"{len(models)} models defined:")
for name in models:
    print(" -", name)


In [ ]:
# Run 5-fold cross-validation for every model and collect R2 / RMSE
cv_results = []

for name, pipeline in models.items():
    scores = cross_validate(
        pipeline, X, y, cv=cv,
        scoring=('r2', 'neg_root_mean_squared_error'),
        n_jobs=-1
    )
    mean_r2 = scores['test_r2'].mean()
    std_r2 = scores['test_r2'].std()
    mean_rmse = -scores['test_neg_root_mean_squared_error'].mean()
    std_rmse = scores['test_neg_root_mean_squared_error'].std()

    cv_results.append({
        'Model': name,
        'Mean CV R2': mean_r2,
        'Std R2': std_r2,
        'Mean CV RMSE': mean_rmse,
        'Std RMSE': std_rmse
    })
    print(f'{name:32s} | R2 = {mean_r2:.4f} (+/-{std_r2:.4f}) | RMSE = {mean_rmse:,.2f}')

cv_results_df = pd.DataFrame(cv_results).sort_values('Mean CV R2', ascending=False).reset_index(drop=True)


In [ ]:
cv_results_df


### Key Insights & Observations

1. **Tree-Based Ensembles Dominate**: 
   Non-linear tree architectures significantly outperform linear formulations. The continuous spatial splits handle the feature interactions much better than static linear coefficients.

2. **Extra Trees Reigns Supreme**: 
   **Extra Trees** achieved the absolute highest variance explanation with an **$R^2$ of 0.9646** and slashed the prediction error down to an **RMSE of 4,277.18**. The random selection of split points likely provided superior regularization against variance compared to standard Random Forest.

3. **Linear Congruence**: 
   Linear, Ridge, and Lasso configurations yield nearly identical metrics ($R^2 \approx 0.9091$). This indicates that L1/L2 penalties are not dropping or heavily shifting variables at default strengths, meaning multi-collinearity is cleanly managed or minor.

4. **ElasticNet Collapse**: 
   ElasticNet penalizes the network far too aggressively out-of-the-box for this specific dataset structure, causing the $R^2$ accuracy to drop steeply to **0.6651**.



## 11. Hyperparameter Tuning

We now tune hyperparameters for **three** models using `GridSearchCV` (small, fast search
spaces, 3-fold CV to keep runtime reasonable): **Ridge Regression**, **Decision Tree**, and
**Random Forest**.


In [ ]:
from sklearn.model_selection import GridSearchCV

tuning_cv = KFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
tuned_results = []
best_estimators = {}


In [ ]:
# --- Tune Ridge Regression ---
ridge_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', Ridge(random_state=RANDOM_STATE))
])

ridge_param_grid = {
    'regressor__alpha': [0.1, 1.0, 5.0, 10.0, 50.0]
}

ridge_search = GridSearchCV(ridge_pipeline, ridge_param_grid, cv=tuning_cv,
                             scoring='r2', n_jobs=-1)
ridge_search.fit(X, y)

print('Best Ridge params:', ridge_search.best_params_)
print('Best Ridge CV R2  :', ridge_search.best_score_)

best_estimators['Ridge Regression (Tuned)'] = ridge_search.best_estimator_
tuned_results.append({'Model': 'Ridge Regression (Tuned)',
                       'Best Params': ridge_search.best_params_,
                       'Best CV R2': ridge_search.best_score_})


In [ ]:
# --- Tune Decision Tree ---
dt_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', DecisionTreeRegressor(random_state=RANDOM_STATE))
])

dt_param_grid = {
    'regressor__max_depth': [8, 12, 16, None],
    'regressor__min_samples_leaf': [1, 5, 10]
}

dt_search = GridSearchCV(dt_pipeline, dt_param_grid, cv=tuning_cv,
                          scoring='r2', n_jobs=-1)
dt_search.fit(X, y)

print('Best Decision Tree params:', dt_search.best_params_)
print('Best Decision Tree CV R2 :', dt_search.best_score_)

best_estimators['Decision Tree (Tuned)'] = dt_search.best_estimator_
tuned_results.append({'Model': 'Decision Tree (Tuned)',
                       'Best Params': dt_search.best_params_,
                       'Best CV R2': dt_search.best_score_})


In [ ]:
from sklearn.model_selection import RandomizedSearchCV

# --- Tune Random Forest  ---
rf_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(random_state=RANDOM_STATE))
])

rf_param_grid = {
    'regressor__n_estimators': [30, 50],
    'regressor__max_depth': [8, 10],
    'regressor__min_samples_leaf': [1, 3]
}

rf_search = RandomizedSearchCV(
    rf_pipeline,
    rf_param_grid,
    n_iter=4,          
    cv=tuning_cv,
    scoring='r2',
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=2
)

rf_search.fit(X, y)

print("Best Random Forest params:", rf_search.best_params_)
print("Best Random Forest CV R2 :", rf_search.best_score_)

best_estimators['Random Forest (Tuned)'] = rf_search.best_estimator_

tuned_results.append({
    'Model': 'Random Forest (Tuned)',
    'Best Params': rf_search.best_params_,
    'Best CV R2': rf_search.best_score_
})

In [ ]:
tuned_results_df = pd.DataFrame(tuned_results).sort_values('Best CV R2', ascending=False).reset_index(drop=True)
tuned_results_df


## 12. Model Comparison

We combine the baseline cross-validation results with the tuned-model results into a single
comparison table to clearly identify the best-performing model.


In [ ]:
# Merge tuned RMSE back in by re-running cross_validate with neg_root_mean_squared_error
# for the tuned estimators, so the comparison table has both R2 and RMSE for everything.
tuned_rmse = []
for name, estimator in best_estimators.items():
    scores = cross_validate(estimator, X, y, cv=cv,
                             scoring=('r2', 'neg_root_mean_squared_error'), n_jobs=-1)
    tuned_rmse.append({
        'Model': name,
        'Mean CV R2': scores['test_r2'].mean(),
        'Std R2': scores['test_r2'].std(),
        'Mean CV RMSE': -scores['test_neg_root_mean_squared_error'].mean(),
        'Std RMSE': scores['test_neg_root_mean_squared_error'].std()
    })

tuned_rmse_df = pd.DataFrame(tuned_rmse)

# Full comparison table: baseline models + tuned models
full_comparison = pd.concat([cv_results_df, tuned_rmse_df], ignore_index=True)
full_comparison = full_comparison.sort_values('Mean CV R2', ascending=False).reset_index(drop=True)
full_comparison[['Model', 'Mean CV R2', 'Mean CV RMSE']]


In [ ]:
best_model_name = full_comparison.loc[0, 'Model']
best_model_r2 = full_comparison.loc[0, 'Mean CV R2']
best_model_rmse = full_comparison.loc[0, 'Mean CV RMSE']

print(f'BEST MODEL: {best_model_name}')
print(f'  Mean CV R2   : {best_model_r2:.4f}')
print(f'  Mean CV RMSE : {best_model_rmse:,.2f}')


In [ ]:
# Visual comparison of models by R2 score
plt.figure(figsize=(9, 6))
plot_df = full_comparison.sort_values('Mean CV R2')
sns.barplot(data=plot_df, y='Model', x='Mean CV R2', palette='crest')
plt.title('Model Comparison — Mean CV R2 (higher is better)')
plt.xlabel('Mean CV R2')
plt.tight_layout()
plt.show()


## 13. Train Final Model on Full Training Data

We now take the best-performing model identified above and retrain it on the **entire** training
dataset (all rows, no CV split held out), so it can make the most informed predictions on the
unseen test set.


In [ ]:
# Select the actual fitted pipeline object for the best model
if best_model_name in best_estimators:
    final_model = best_estimators[best_model_name]
    # already fitted on full X, y during GridSearchCV refit=True (default),
    # but we refit explicitly here for clarity/safety.
    final_model.fit(X, y)
else:
    final_model = models[best_model_name]
    final_model.fit(X, y)

print(f'Final model trained: {best_model_name}')
final_model


## 14. Kaggle Submission

We use the final trained model to predict prices on the test set, and save the predictions in
the format required by `sample_submission.csv`.


In [ ]:
X_test_final = test_fe.copy()  # test_fe already has id/flight dropped by engineer_features

test_predictions = final_model.predict(X_test_final)

# Prices cannot be negative — clip any stray negative predictions at 0 as a safety net
test_predictions = np.clip(test_predictions, a_min=0, a_max=None)

print('Generated', len(test_predictions), 'predictions.')
print('Prediction summary statistics:')
print(pd.Series(test_predictions).describe())


In [ ]:
submission = sample_submission.copy()
submission['price'] = test_predictions

submission.to_csv('submission.csv', index=False)

print('submission.csv saved successfully.')
submission.head()


## 15. Feature Importance

Finally, we inspect which features the final model relied on most. If the selected model is
tree-based, we use its built-in `feature_importances_`. Otherwise (e.g. a linear model), we fall
back to **permutation importance**, which works for any model type.


In [ ]:
from sklearn.inspection import permutation_importance

# Recover the fitted preprocessor and regressor from the final pipeline
fitted_preprocessor = final_model.named_steps['preprocessor']
fitted_regressor = final_model.named_steps['regressor']

# Get the expanded feature names produced by the ColumnTransformer (numeric + one-hot columns)
feature_names = fitted_preprocessor.get_feature_names_out()

if hasattr(fitted_regressor, 'feature_importances_'):
    importances = fitted_regressor.feature_importances_
    importance_df = pd.DataFrame({
        'feature': feature_names,
        'importance': importances
    }).sort_values('importance', ascending=False).head(15)

    plt.figure(figsize=(9, 7))
    sns.barplot(data=importance_df, y='feature', x='importance', palette='mako')
    plt.title(f'Top 15 Feature Importances — {best_model_name}')
    plt.xlabel('Importance')
    plt.tight_layout()
    plt.show()

else:
    print('Selected model has no built-in feature_importances_. Using permutation importance instead.')
    perm_result = permutation_importance(
        final_model, X, y, n_repeats=5, random_state=RANDOM_STATE, n_jobs=-1
    )
    perm_df = pd.DataFrame({
        'feature': X.columns,
        'importance': perm_result.importances_mean
    }).sort_values('importance', ascending=False)

    plt.figure(figsize=(9, 7))
    sns.barplot(data=perm_df, y='feature', x='importance', palette='mako')
    plt.title(f'Permutation Feature Importance — {best_model_name}')
    plt.xlabel('Mean Decrease in Score')
    plt.tight_layout()
    plt.show()


## 16. Conclusion

**Preprocessing performed:**
- Identified and explicitly listed data types for every column.
- Identified missing values in both `train.csv` and `test.csv` and handled them via
  median imputation (numerical) and most-frequent imputation (categorical), inside
  leak-free scikit-learn pipelines.
- Checked for and removed duplicate rows in the training set.
- Analysed outliers via boxplots + the IQR method, and made a deliberate decision to
  **retain** them, since they represent genuine Business-class / multi-stop flight behaviour
  rather than data errors.
- Performed feature engineering: dropped the non-informative `id` column and the
  high-cardinality/partially-corrupted `flight` column.
- Scaled numerical features with `StandardScaler` and one-hot encoded categorical features
  with `OneHotEncoder`, explaining the rationale for each.

**Models compared:** 9 scikit-learn regression models were trained and evaluated with 5-fold
cross-validation — Linear Regression, Ridge, Lasso, ElasticNet, Polynomial Regression (degree 2),
Decision Tree, Random Forest, Extra Trees, and HistGradientBoosting — using R² and RMSE as
evaluation metrics.

**Hyperparameter tuning:** GridSearchCV was used to tune Ridge Regression, Decision Tree, and
Random Forest with small, fast search spaces (3-fold CV).

**Best model:** Based on the model comparison table, **`{BEST_MODEL_NAME}`** achieved the
highest mean cross-validated R² and was selected as the final model. It was retrained on the
full training dataset and used to generate predictions on `test.csv`.

**Kaggle submission:** Final predictions were saved as `submission.csv` in the exact format of
`sample_submission.csv` (columns `id`, `price`), ready for upload to Kaggle.

**Feature importance:** The most influential features driving ticket price were examined and
visualised, with `class` (Business vs. Economy) standing out as expected from the EDA.
